# 03 — Longitudinal matching, stability, ambiguity, and event-level join

This notebook covers Tasks 10–16. It generates rankings diagnostically but **does not create or save a final crosswalk**. All raw data are read directly from `data/`; all derived matrices and tables remain in memory and outputs are embedded in this notebook.

### Evidence definitions

For each Ticket vehicle-time observation and Mobility candidate:

- `same`: at least one candidate telemetry reading within the tolerance is on the Ticket route;
- `conflict`: telemetry exists within tolerance, but none is on the Ticket route;
- `missing`: no candidate telemetry within tolerance.

Ambiguous GPS id/30-second bins that contain multiple routes are retained as all observed route possibilities. This is conservative for conflict detection and is reported. Ticket observations are vehicle-minute route states represented by their median boarding second within that minute. The 30-second grid makes the ±30/60/90/120-second sensitivity explicit.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import re, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

HERE = Path.cwd().resolve()
PROJECT = HERE.parent if HERE.name == "mob_tick_matching_robustness_check" else HERE
DATA = PROJECT / "data"
MOB_FILES = sorted((DATA / "mobility_data").glob("*.csv"))
TIX_FILES = sorted((DATA / "ticket_data").glob("*.csv"))
BAD_DATES = {"2026-03-18", "2026-03-19", "2026-03-20", "2026-03-22", "2026-03-28"}
RELIABLE_FILES_M = [p for p in MOB_FILES if p.stem not in BAD_DATES]
RELIABLE_DATES = {p.stem for p in RELIABLE_FILES_M}
RELIABLE_FILES_T = [p for p in TIX_FILES if p.stem in RELIABLE_DATES]

def canon_route(x):
    """Conservative formatting normalization; does not remove leading zeroes or punctuation."""
    if pd.isna(x): return pd.NA
    s = str(x).strip()
    return re.sub(r"^(\d+)\.0$", r"\1", s)

def canon_name(x):
    if pd.isna(x): return pd.NA
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c)).upper().strip()
    return re.sub(r"\s+", " ", s)

print(f"Project: {PROJECT}")
print(f"Mobility files: {len(MOB_FILES)}; Ticket files: {len(TIX_FILES)}; reliable overlapping dates: {len(RELIABLE_DATES)}")

Project: /Users/pirin/Desktop/Under Grad Research/Netmob2026_data_challenge
Mobility files: 19; Ticket files: 31; reliable overlapping dates: 16


In [2]:
# Establish Ticket key necessity using raw files.
vc=defaultdict(set)
for p in TIX_FILES:
    d=pd.read_csv(p,usecols=["vehicle_number","company_number"],dtype="string").drop_duplicates()
    for v,c in d.dropna().itertuples(index=False): vc[v].add(c)
USE_COMPOSITE=any(len(x)>1 for x in vc.values())
def ticket_key(df): return (df.company_number.fillna("<NA>")+"::"+df.vehicle_number.fillna("<NA>")) if USE_COMPOSITE else df.vehicle_number
print("Ticket key:","(company_number, vehicle_number)" if USE_COMPOSITE else "vehicle_number")

# Load reliable observations only, preserving route strings. No files are written.
mob=[]; tix=[]
for p in RELIABLE_FILES_M:
    d=pd.read_csv(p,usecols=["id","timestamp","lineId"],dtype="string"); ts=pd.to_datetime(d.timestamp,errors="coerce")
    mob.append(pd.DataFrame({"date":p.stem,"id":d.id,"time":ts,"route":d.lineId.map(canon_route)}).dropna())
for p in RELIABLE_FILES_T:
    d=pd.read_csv(p,usecols=["transaction_date","vehicle_number","company_number","route_name"],dtype="string")
    ts=pd.to_datetime(d.transaction_date,errors="coerce",utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
    tix.append(pd.DataFrame({"date":p.stem,"vehicle_key":ticket_key(d),"time":ts,"route":d.route_name.map(canon_route)}).dropna())
mob_raw=pd.concat(mob,ignore_index=True); tix_raw=pd.concat(tix,ignore_index=True)

# Collapse Ticket transactions into vehicle-minute route states; retain median event time and boarding count.
tix_raw["minute"]=tix_raw.time.dt.floor("min")
tmin=tix_raw.groupby(["date","vehicle_key","minute","route"]).agg(time=("time","median"),boarding_count=("time","size")).reset_index()
ticket_multi=tmin.groupby(["date","vehicle_key","minute"]).route.nunique().rename("n_routes").reset_index().query("n_routes>1")
print({"raw_ticket_transactions":len(tix_raw),"ticket_vehicle_minute_routes":len(tmin),"ticket_multi_route_vehicle_minutes":len(ticket_multi),"raw_mobility_observations":len(mob_raw)})
display(ticket_multi)

Ticket key: (company_number, vehicle_number)


{'raw_ticket_transactions': 3662195, 'ticket_vehicle_minute_routes': 1583606, 'ticket_multi_route_vehicle_minutes': 6, 'raw_mobility_observations': 13035355}


,date,vehicle_key,minute,n_routes
537503,2026-03-16,56::22104,2026-03-16 11:32:00,2
593809,2026-03-17,3::11069,2026-03-17 02:48:00,2
593810,2026-03-17,3::11069,2026-03-17 02:49:00,2
1156008,2026-03-26,56::2144,2026-03-26 13:28:00,2
1427477,2026-03-30,56::2152,2026-03-30 15:08:00,2
1545068,2026-03-31,56::2156,2026-03-31 14:39:00,2


In [3]:
# Encode 30-second bins into a dense time × Mobility-ID route-state matrix.
all_routes=sorted(set(tmin.route)|set(mob_raw.route)); route_code={r:i for i,r in enumerate(all_routes)}
mob_ids=sorted(mob_raw.id.unique()); midx={m:i for i,m in enumerate(mob_ids)}
tkeys=sorted(tmin.vehicle_key.unique())
origin=min(mob_raw.time.min(),tmin.time.min()-pd.Timedelta(hours=3)).floor("30s")
mob_raw["bin"]=((mob_raw.time-origin).dt.total_seconds()//30).astype("int32")
n_bins=int(max(mob_raw.bin.max(),((tmin.time+pd.Timedelta(hours=3)-origin).dt.total_seconds()//30).max())+2); n_mob=len(mob_ids)
state=np.full((n_bins,n_mob),-1,dtype=np.int16)
u=mob_raw[["bin","id","route"]].drop_duplicates(); br=u.bin.to_numpy(); bc=u.id.map(midx).to_numpy(); rv=u.route.map(route_code).to_numpy(dtype=np.int16)
state[br,bc]=rv
amb=u.groupby(["bin","id"]).route.nunique(); amb=amb[amb>1]
ambiguous_routes=defaultdict(dict)
if len(amb):
    keys=set(amb.index)
    for (b,m),g in u.groupby(["bin","id"]):
        if (b,m) in keys:
            c=midx[m]; state[int(b),c]=-2; ambiguous_routes[int(b)][c]={route_code[r] for r in g.route}
print({"30s_bins":n_bins,"mobility_ids":n_mob,"ambiguous_id_30s_bins":len(amb)})

def apply_clock_shift(frame,minutes):
    z=frame.copy(); z["bin"]=(((z.time+pd.Timedelta(minutes=minutes))-origin).dt.total_seconds()//30).astype("int32"); return z

def score_one(obs, tolerance_seconds=60, mob_subset=None):
    """Return counts for all or selected Mobility IDs. Any same-route hit wins over conflict within window."""
    cols=np.arange(n_mob) if mob_subset is None else np.asarray(mob_subset,dtype=int)
    same=np.zeros(len(cols),dtype=np.int64); comparable=np.zeros(len(cols),dtype=np.int64)
    radius=int(np.ceil(tolerance_seconds/30))
    colpos={int(c):i for i,c in enumerate(cols)} if mob_subset is not None else None
    bins=obs.bin.to_numpy(dtype=int); rcs=obs.route.map(route_code).to_numpy(dtype=np.int16)
    seen=np.zeros((len(obs),len(cols)),dtype=bool); hit=np.zeros_like(seen)
    for off in range(-radius,radius+1):
        q=bins+off; valid=(q>=0)&(q<n_bins)
        arr=np.full((len(obs),len(cols)),-1,dtype=np.int16)
        arr[valid]=state[q[valid]][:,cols]
        seen |= arr!=-1; hit |= arr==rcs[:,None]
        # Restore all route possibilities for rare ambiguous id-bins.
        for row in np.flatnonzero(valid):
            for c,rs in ambiguous_routes.get(int(q[row]),{}).items():
                pos=c if colpos is None else colpos.get(c)
                if pos is not None and rcs[row] in rs: hit[row,pos]=True
    comparable=seen.sum(axis=0); same=hit.sum(axis=0)
    conflict=comparable-same; missing=len(obs)-comparable
    return same,conflict,missing

def all_scores(frame,tol=60):
    rows=[]
    for k,g in frame.groupby("vehicle_key",sort=False):
        same,conf,miss=score_one(g,tol)
        for j,m in enumerate(mob_ids):
            if same[j]>0 or conf[j]>0:
                rows.append((k,m,int(same[j]),int(conf[j]),int(miss[j])))
    x=pd.DataFrame(rows,columns=["ticket_vehicle","mobility_id","N_same","N_conflict","N_missing"])
    x["compatibility"]=x.N_same/(x.N_same+x.N_conflict)
    return x

display(Markdown("## D. Longitudinal clock-offset diagnostic (four representative reliable dates)"))
sample_dates=sorted(RELIABLE_DATES)[::max(1,len(RELIABLE_DATES)//4)][:4]
offset_quality=[]
for off in [-180,-60,0,60,180]:
    so=all_scores(apply_clock_shift(tmin[tmin.date.isin(sample_dates)],off),60)
    top=so.sort_values(["ticket_vehicle","compatibility","N_same"],ascending=[True,False,False]).groupby("ticket_vehicle").head(1)
    offset_quality.append({"ticket_shift_minutes":off,"sample_dates":" | ".join(sample_dates),"median_best_compatibility":top.compatibility.median(),"median_best_N_same":top.N_same.median(),"total_best_N_same":top.N_same.sum(),"pct_best_ge_0.95":100*(top.compatibility>=.95).mean()})
offset_quality=pd.DataFrame(offset_quality); display(offset_quality)
SELECTED_SHIFT=int(offset_quality.sort_values(["total_best_N_same","median_best_compatibility"],ascending=False).iloc[0].ticket_shift_minutes)
print("Empirically selected Ticket clock shift for the primary diagnostic:",SELECTED_SHIFT,"minutes")
tmin=apply_clock_shift(tmin,SELECTED_SHIFT)

sens=[]; score_by_tol={}
for tol in [30,60,90,120]:
    s=all_scores(tmin,tol); score_by_tol[tol]=s
    ranked=s.sort_values(["ticket_vehicle","compatibility","N_same"],ascending=[True,False,False])
    top=ranked.groupby("ticket_vehicle").head(1)
    sens.append({"tolerance_seconds":tol,"pairs_evaluated":len(s),"median_best_compatibility":top.compatibility.median(),"median_best_N_same":top.N_same.median(),"pct_best_ge_0.95":100*(top.compatibility>=.95).mean()})
display(Markdown("## F. Tolerance sensitivity")); display(pd.DataFrame(sens))

{'30s_bins': 61198, 'mobility_ids': 527, 'ambiguous_id_30s_bins': 668}


## D. Longitudinal clock-offset diagnostic (four representative reliable dates)

,ticket_shift_minutes,sample_dates,median_best_compatibility,median_best_N_same,total_best_N_same,pct_best_ge_0.95
0,-180,2026-03-11 | 2026-03-15 | 2026-03-23 | 2026-03-27,1.0,180.0,115434,84.643510
1,-60,2026-03-11 | 2026-03-15 | 2026-03-23 | 2026-03-27,1.0,218.0,137357,85.191956
2,0,2026-03-11 | 2026-03-15 | 2026-03-23 | 2026-03-27,1.0,237.0,152170,85.374771
3,60,2026-03-11 | 2026-03-15 | 2026-03-23 | 2026-03-27,1.0,259.0,169603,85.009141
4,180,2026-03-11 | 2026-03-15 | 2026-03-23 | 2026-03-27,1.0,360.0,215499,87.934186


Empirically selected Ticket clock shift for the primary diagnostic: 180 minutes


## F. Tolerance sensitivity

,tolerance_seconds,pairs_evaluated,median_best_compatibility,median_best_N_same,pct_best_ge_0.95
0,30,292659,1.0,1385.0,75.395431
1,60,292669,1.0,1398.0,75.395431
2,90,292680,1.0,1404.0,75.395431
3,120,292683,1.0,1419.0,75.219684


In [4]:
# Main ranking at ±60 seconds, with days supported and second-best gap.
s=score_by_tol[60].copy()
support=[]
for date,gdate in tmin.groupby("date"):
    sd=all_scores(gdate,60)
    if len(sd):
        support.append(sd.loc[sd.N_same>0,["ticket_vehicle","mobility_id"]].assign(date=date))
support=pd.concat(support,ignore_index=True) if support else pd.DataFrame(columns=["ticket_vehicle","mobility_id","date"])
days=support.groupby(["ticket_vehicle","mobility_id"]).date.nunique().rename("days_supported").reset_index()
s=s.merge(days,on=["ticket_vehicle","mobility_id"],how="left").fillna({"days_supported":0})
s=s.sort_values(["ticket_vehicle","compatibility","N_same"],ascending=[True,False,False])
s["candidate_rank"]=s.groupby("ticket_vehicle").cumcount()+1
second=s.query("candidate_rank==2").set_index("ticket_vehicle").compatibility
s["second_best_gap"]=s.apply(lambda r:r.compatibility-second.get(r.ticket_vehicle,np.nan) if r.candidate_rank==1 else np.nan,axis=1)
cols=["ticket_vehicle","candidate_rank","mobility_id","N_same","N_conflict","N_missing","compatibility","second_best_gap","days_supported"]
top_candidates=s.query("candidate_rank<=5")[cols]
display(Markdown("## F. Top five candidate pairs for every Ticket vehicle")); display(top_candidates)

# Conservative diagnostic confidence gate; it is not a final crosswalk.
best=s.query("candidate_rank==1").copy()
best["high_confidence_gate"]=(best.compatibility>=.98)&(best.second_best_gap>=.10)&(best.N_same>=100)&(best.days_supported>=3)
display(Markdown("### Diagnostic confidence-gate counts")); display(best.high_confidence_gate.value_counts(dropna=False).rename_axis("passes").to_frame("vehicles"))
display(Markdown("### Ambiguous/problematic examples (low gap, low compatibility, or weak support)")); display(best.loc[~best.high_confidence_gate,cols].sort_values(["second_best_gap","N_same"]).head(100))

## F. Top five candidate pairs for every Ticket vehicle

,ticket_vehicle,candidate_rank,mobility_id,N_same,N_conflict,N_missing,compatibility,second_best_gap,days_supported
0,11::14001,1,102866,0,704,2478,0.0,0.0,0.0
1,11::14001,2,103056,0,1226,1956,0.0,NaN,0.0
2,11::14001,3,103697,0,1942,1240,0.0,NaN,0.0
3,11::14001,4,103837,0,2075,1107,0.0,NaN,0.0
4,11::14001,5,103838,0,2409,773,0.0,NaN,0.0
...,...,...,...,...,...,...,...,...,...
260394,56::24142,1,108650,1356,0,852,1.0,0.0,8.0
260623,56::24142,2,69181,1333,0,875,1.0,NaN,8.0
260720,56::24142,3,77793,1326,0,882,1.0,NaN,8.0
260616,56::24142,4,69166,1297,0,911,1.0,NaN,8.0


### Diagnostic confidence-gate counts

,vehicles
passes,
False,464
True,105


### Ambiguous/problematic examples (low gap, low compatibility, or weak support)

,ticket_vehicle,candidate_rank,mobility_id,N_same,N_conflict,N_missing,compatibility,second_best_gap,days_supported
0,11::14001,1,102866,0,704,2478,0.0,0.0,0.0
3660,22::11028,1,102866,0,474,3045,0.0,0.0,0.0
261368,22::11075,1,102866,0,3,46,0.0,0.0,0.0
6257,22::11078,1,102866,0,368,1403,0.0,0.0,0.0
6778,22::11079,1,102866,0,504,2134,0.0,0.0,0.0
36591,27::15067,1,102866,0,9,377,0.0,0.0,0.0
67259,3::11001,1,102866,0,876,2569,0.0,0.0,0.0
272171,3::11102,1,102866,0,400,2161,0.0,0.0,0.0
93389,3::11104,1,102866,0,791,2858,0.0,0.0,0.0
96546,3::11124,1,102866,0,646,2798,0.0,0.0,0.0


In [5]:
# Bidirectional uniqueness and collisions.
gps_pref=s.sort_values(["mobility_id","compatibility","N_same"],ascending=[True,False,False]).groupby("mobility_id").head(1)[["mobility_id","ticket_vehicle"]].rename(columns={"ticket_vehicle":"gps_preferred_ticket"})
best=best.merge(gps_pref,on="mobility_id",how="left")
best["reciprocal_top1"]=best.ticket_vehicle.eq(best.gps_preferred_ticket)
coll=best.groupby("mobility_id").agg(ticket_top1_count=("ticket_vehicle","size"),ticket_vehicles=("ticket_vehicle",lambda x:" | ".join(x))).query("ticket_top1_count>1").reset_index()
display(Markdown("## Bidirectional uniqueness")); print(best.reciprocal_top1.value_counts(dropna=False)); display(Markdown("### GPS top-1 collisions (not greedily resolved)")); display(coll)

## Bidirectional uniqueness

reciprocal_top1
True     359
False    210
Name: count, dtype: int64


### GPS top-1 collisions (not greedily resolved)

,mobility_id,ticket_top1_count,ticket_vehicles
0,102866,21,11::14001 | 22::11028 | 22::11075 | 22::11078 ...
1,103697,6,2::13030 | 3::21004 | 3::21006 | 3::21029 | 56...
2,105885,2,56::2112 | 56::2116
3,106314,2,11::14017 | 2::13005
4,108650,3,56::24105 | 56::24109 | 56::24142
5,112005,2,27::15029 | 27::15071
6,112006,2,27::15041 | 27::15070
7,112047,2,27::15066 | 27::15166
8,124778,2,2::13058 | 2::13059
9,130628,2,27::15040 | 27::15085


In [6]:
# Day-level best candidates, consistency, and change-point clues.
daytops=[]
for date,gdate in tmin.groupby("date"):
    sd=all_scores(gdate,60).sort_values(["ticket_vehicle","compatibility","N_same"],ascending=[True,False,False]).groupby("ticket_vehicle").head(1)
    sd["date"]=date; daytops.append(sd)
daytops=pd.concat(daytops,ignore_index=True)
overall=best.set_index("ticket_vehicle").mobility_id
cons=[]
for k,g in daytops.groupby("ticket_vehicle"):
    supported=g[g.N_same>=10]
    ov=overall.get(k); same_n=int((supported.mobility_id==ov).sum())
    seq=" | ".join(f"{d}:{m}" for d,m in supported.sort_values("date")[["date","mobility_id"]].itertuples(index=False))
    cons.append({"ticket_vehicle":k,"best_candidate_overall":ov,"number_of_supported_days":len(supported),"number_of_days_same_candidate_is_best":same_n,"consistency_rate":same_n/len(supported) if len(supported) else np.nan,"daily_best_sequence":seq})
consistency=pd.DataFrame(cons).sort_values(["consistency_rate","number_of_supported_days"])
display(Markdown("## G. Day-to-day stability and possible change points")); display(consistency)
stable_eval=consistency.merge(best[["ticket_vehicle","high_confidence_gate","reciprocal_top1"]],on="ticket_vehicle",how="left")
stable_eval["confidence_group"]=np.where(stable_eval.high_confidence_gate & stable_eval.reciprocal_top1,"high-confidence reciprocal","other")
display(stable_eval.groupby("confidence_group").agg(vehicles=("ticket_vehicle","size"),median_supported_days=("number_of_supported_days","median"),median_consistency=("consistency_rate","median"),pct_consistency_1=("consistency_rate",lambda x:100*(x==1).mean()),pct_consistency_ge_0_8=("consistency_rate",lambda x:100*(x>=.8).mean())).reset_index())

## G. Day-to-day stability and possible change points

,ticket_vehicle,best_candidate_overall,number_of_supported_days,number_of_days_same_candidate_is_best,consistency_rate,daily_best_sequence
254,3::12025,99006,1,0,0.0,2026-03-17:99005
12,22::11027,97136,2,0,0.0,2026-03-16:97094 | 2026-03-23:97094
145,2::13055,99155,2,0,0.0,2026-03-14:99113 | 2026-03-21:99164
228,3::11126,97171,2,0,0.0,2026-03-16:97087 | 2026-03-24:97130
280,56::2011,96725,2,0,0.0,2026-03-13:72634 | 2026-03-31:72630
...,...,...,...,...,...,...
337,56::2068,102866,0,0,NaN,
338,56::2069,83518,0,0,NaN,
339,56::2070,102866,0,0,NaN,
347,56::2086,103697,0,0,NaN,


,confidence_group,vehicles,median_supported_days,median_consistency,pct_consistency_1,pct_consistency_ge_0_8
0,high-confidence reciprocal,90,11.0,0.500000,5.555556,14.444444
1,other,479,9.0,0.303846,4.384134,7.933194


In [7]:
# Quantify fingerprint identifiability over several hours, one day, 3-day blocks, and whole period.
def segment_summary(frame,label,segmenter):
    vals=[]
    z=frame.copy(); z["segment"]=segmenter(z)
    for seg,g in z.groupby("segment"):
        sc=all_scores(g,60).sort_values(["ticket_vehicle","compatibility","N_same"],ascending=[True,False,False])
        first=sc.groupby("ticket_vehicle").nth(0).reset_index(); second_=sc.groupby("ticket_vehicle").nth(1).reset_index()[["ticket_vehicle","compatibility"]].rename(columns={"compatibility":"second"})
        q=first.merge(second_,on="ticket_vehicle",how="left"); q["gap"]=q.compatibility-q.second
        vals.append(q.assign(segment=str(seg)))
    q=pd.concat(vals,ignore_index=True)
    return {"horizon":label,"segments":q.segment.nunique(),"vehicle_segment_estimates":len(q),"median_best_compatibility":q.compatibility.median(),"median_gap":q.gap.median(),"pct_clear_(comp>=.95,gap>=.10,Nsame>=20)":100*((q.compatibility>=.95)&(q.gap>=.10)&(q.N_same>=20)).mean()}

base=pd.Timestamp(sorted(RELIABLE_DATES)[0])
horizons=[]
horizons.append(segment_summary(tmin,"4 hours",lambda z:z.date+"_"+(z.time.dt.hour//4).astype(str)))
horizons.append(segment_summary(tmin,"1 day",lambda z:z.date))
horizons.append(segment_summary(tmin,"3-day blocks",lambda z:((pd.to_datetime(z.date)-base).dt.days//3).astype(str)))
# Whole-period results reuse the main score.
tmp=best.copy(); horizons.append({"horizon":"whole reliable period","segments":1,"vehicle_segment_estimates":len(tmp),"median_best_compatibility":tmp.compatibility.median(),"median_gap":tmp.second_best_gap.median(),"pct_clear_(comp>=.95,gap>=.10,Nsame>=20)":100*((tmp.compatibility>=.95)&(tmp.second_best_gap>=.10)&(tmp.N_same>=20)).mean()})
display(Markdown("## Route-switching sequence as a longitudinal fingerprint")); display(pd.DataFrame(horizons))

## Route-switching sequence as a longitudinal fingerprint

,horizon,segments,vehicle_segment_estimates,median_best_compatibility,median_gap,"pct_clear_(comp>=.95,gap>=.10,Nsame>=20)"
0,4 hours,92,29932,1.0,0.000000,0.825204
1,1 day,16,6898,1.0,0.000000,2.391998
2,3-day blocks,7,3371,1.0,0.000000,5.339662
3,whole reliable period,1,569,1.0,0.033036,23.374341


In [8]:
# Diagnostic assignment suitability. Do not execute Hungarian assignment unless independent top-1 is sufficiently one-to-one.
collision_rate=100*best.mobility_id.duplicated(keep=False).mean()
print({"ticket_top1_collision_vehicle_pct":collision_rate,"reciprocal_top1_pct":100*best.reciprocal_top1.mean()})
if collision_rate==0:
    display(Markdown("Independent top-1 is already one-to-one, so Hungarian assignment would not add diagnostic value and is not run."))
else:
    display(Markdown("Top-1 collisions exist. A global assignment could enforce uniqueness, but should not be treated as evidence unless pairwise gaps and day stability are already strong. It is deliberately not run because forced assignment can manufacture matches for ambiguous vehicles."))

{'ticket_top1_collision_vehicle_pct': np.float64(41.30052724077329), 'reciprocal_top1_pct': np.float64(63.09314586994728)}


Top-1 collisions exist. A global assignment could enforce uniqueness, but should not be treated as evidence unless pairwise gaps and day stability are already strong. It is deliberately not run because forced assignment can manufacture matches for ambiguous vehicles.

In [9]:
# Event-level nearest-time join for pairs passing the diagnostic gate AND reciprocal preference.
hc=best[best.high_confidence_gate & best.reciprocal_top1][["ticket_vehicle","mobility_id"]].copy()
mapping=dict(zip(hc.ticket_vehicle,hc.mobility_id)); diffs=[]; joined=[]
for date in sorted(RELIABLE_DATES):
    tf=next(p for p in RELIABLE_FILES_T if p.stem==date); mf=next(p for p in RELIABLE_FILES_M if p.stem==date)
    td=pd.read_csv(tf,usecols=["transaction_date","vehicle_number","company_number","route_name","route_detail_id"],dtype="string")
    td["ticket_vehicle"]=ticket_key(td); td=td[td.ticket_vehicle.isin(mapping)].copy()
    td["ticket_time"]=pd.to_datetime(td.transaction_date,errors="coerce",utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)+pd.Timedelta(minutes=SELECTED_SHIFT); td["match_id"]=td.ticket_vehicle.map(mapping); td["ticket_route"]=td.route_name.map(canon_route)
    gd=pd.read_csv(mf,usecols=["timestamp","id","lineId","tripId","direction","headsign"],dtype="string"); gd=gd[gd.id.isin(set(mapping.values()))].copy(); gd["gps_time"]=pd.to_datetime(gd.timestamp,errors="coerce"); gd["gps_route"]=gd.lineId.map(canon_route); gd=gd.rename(columns={"id":"match_id"})
    # merge_asof requires global ordering on the join key.
    td=td.dropna(subset=["ticket_time"]).sort_values("ticket_time"); gd=gd.dropna(subset=["gps_time"]).sort_values("gps_time")
    if len(td) and len(gd):
        day_join=[]
        for match_id,left in td.groupby("match_id"):
            right=gd[gd.match_id.astype(str)==str(match_id)].drop(columns="match_id").sort_values("gps_time")
            if len(right):
                day_join.append(pd.merge_asof(left.sort_values("ticket_time"),right,left_on="ticket_time",right_on="gps_time",direction="nearest",tolerance=pd.Timedelta("10min")))
        if day_join:
            j=pd.concat(day_join,ignore_index=True); j["abs_seconds"]=(j.gps_time-j.ticket_time).abs().dt.total_seconds(); j["route_agree"]=j.ticket_route.eq(j.gps_route)
            diffs.append(j.abs_seconds.dropna().to_numpy()); joined.append(j[["ticket_vehicle","match_id","route_detail_id","ticket_route","gps_route","tripId","direction","headsign","abs_seconds","route_agree"]])
event_join=pd.concat(joined,ignore_index=True) if joined else pd.DataFrame()
display(Markdown(f"## I. Event-level join diagnostic ({len(hc)} high-confidence reciprocal pairs)"))
if len(event_join):
    x=event_join.abs_seconds.dropna()
    display(pd.DataFrame([{"joined_transactions":len(x),"unjoined_transactions":int(event_join.abs_seconds.isna().sum()),"median_s":x.median(),"mean_s":x.mean(),"p90_s":x.quantile(.90),"p95_s":x.quantile(.95),"p99_s":x.quantile(.99),"pct_within_15s":100*(x<=15).mean(),"pct_within_30s":100*(x<=30).mean(),"pct_within_60s":100*(x<=60).mean(),"pct_within_120s":100*(x<=120).mean(),"route_agreement_pct":100*event_join.loc[event_join.abs_seconds.notna(),"route_agree"].mean()}]))
    # route_detail diagnostic only on close, route-agree joins; no equality assumptions.
    q=event_join[(event_join.abs_seconds<=60)&event_join.route_agree]
    rd=q.groupby("route_detail_id").agg(joined_events=("tripId","size"),distinct_tripId=("tripId","nunique"),distinct_direction=("direction","nunique"),distinct_headsign=("headsign","nunique"),dominant_tripId_share=("tripId",lambda x:x.value_counts(normalize=True).iloc[0] if x.notna().any() else np.nan),dominant_direction_share=("direction",lambda x:x.value_counts(normalize=True).iloc[0] if x.notna().any() else np.nan)).reset_index()
    display(Markdown("### `route_detail_id` association with Mobility trip/direction/headsign among close route-agree joins")); display(rd.sort_values(["distinct_direction","distinct_tripId"],ascending=[False,False]))
else:
    display(Markdown("No pairs passed the conservative gate; event-level localization is therefore not claimed."))

## I. Event-level join diagnostic (90 high-confidence reciprocal pairs)

,joined_transactions,unjoined_transactions,median_s,mean_s,p90_s,p95_s,p99_s,pct_within_15s,pct_within_30s,pct_within_60s,pct_within_120s,route_agreement_pct
0,418266,61476,4.0,20.034672,8.0,118.0,428.0,91.93408,92.507639,93.404197,95.055061,99.36165


### `route_detail_id` association with Mobility trip/direction/headsign among close route-agree joins

,route_detail_id,joined_events,distinct_tripId,distinct_direction,distinct_headsign,dominant_tripId_share,dominant_direction_share
18,2728,36643,224,2,2,0.021423,0.575035
19,2729,24589,203,2,2,0.036642,0.516328
17,2727,15076,176,2,2,0.026665,0.613425
16,2726,11535,161,2,2,0.041959,0.616558
24,2736,6619,107,2,2,0.041094,0.811905
23,2735,11820,99,2,2,0.039340,0.670305
39,84,13577,95,2,2,0.042867,0.658982
21,2733,3193,90,2,4,0.044472,0.758847
20,2732,5153,89,2,2,0.036484,0.714729
38,75,17287,89,2,2,0.051079,0.610922


## Notebook 03 interpretation guide

- A highest score alone is not a match. Require strong compatibility, a meaningful second-best gap, adequate `N_same`, multi-day support, reciprocal top-1 preference, and day stability.
- Low `N_same` is weak evidence even with compatibility 1.0.
- `N_missing` is never counted in the compatibility denominator.
- Hardware replacement is plausible when a well-supported daily-best sequence changes cleanly and persistently; such cases should use time-varying mappings rather than a forced monthly mapping.
- Event-level GPS attachment is credible only for the conservative high-confidence reciprocal subset and only if nearest-time gaps and route agreement are strong.